# 16 — Walidacja międzyrynkowa na S&P 500 / innym indeksie akcji
To jedyny z nowych notatników, który pozwala empirycznie odnieść się do
tytułu "Stock Markets". Wynik **transferowy**: model z BTC (notatnik 14)
jest stosowany na indeksie akcji BEZ uczenia ani strojenia na tym indeksie.

Uwaga: to mocny test zmiany domeny, ale BTC (7 dni/tydzień) oraz indeksy
(sesje giełdowe) mają różną częstotliwość obserwacji. Horyzonty i okna
muszą być jawnie opisane w publikacji.

In [ ]:
import sys as _sys
from pathlib import Path as _ResearchPath
_sys.path.insert(0,str(_ResearchPath('.').resolve()/'src'))
from lppls_research import TemporalAvailabilityValidator
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from bubble_tools import (load_prices,monte_carlo_bsadf_reference,
                          build_daily_features,classification_stats)

ROOT=Path('.')
OUT=ROOT/'16_wyniki_crossmarket'
OUT.mkdir(exist_ok=True,parents=True)

# Domyślnie wymaga rzeczywistego pliku stock_index_prices.csv (Date + Close/price).
STOCK_FILE=ROOT/'stock_index_prices.csv'
STOCK_FITS=ROOT/'stock_index_lppls_all_fits.csv'
BTC_TRANSFER=ROOT/'14_wyniki_bubble'/'14_transfer_BTC_XGB_PriceLPPLS.joblib'

TICKER='^GSPC'  # informacyjnie, dla opcjonalnego pobrania danych
DOWNLOAD_MARKET=False # zmień na True, jeśli świadomie chcesz pobrać Yahoo
GENERATE_STOCK_LPPLS=False # True uruchamia nową, kosztowną estymację
STOCK_FIT_LOOKBACKS=(90,120,180) # liczba SESJI, nie dni kalendarzowych
STOCK_FIT_STEP=10
STOCK_FIT_RESTARTS=3
STOCK_FIT_MAX_WINDOWS=None # limit tylko do pilota; NIE używać do tabel końcowych
SEED=42

# BSADF z kalendarzem sesji giełdowych, przy tych samych parametrach numerycznych;
# interpretacja porównania z BTC jest zależna od wybranej konwencji okien.
MIN_WINDOW=120
END_STEP=7
START_STEP=14
MC_REPS=199

if DOWNLOAD_MARKET and not STOCK_FILE.exists():
    # %pip install yfinance
    import yfinance as yf
    raw=yf.download(TICKER,start='2015-01-01',end='2025-01-01',
                    auto_adjust=True,progress=False,interval='1d')
    if raw.empty:raise RuntimeError('Pobieranie indeksu nie powiodło się')
    if isinstance(raw.columns,pd.MultiIndex):raw.columns=raw.columns.get_level_values(0)
    raw[['Close']].reset_index().rename(columns={'Close':'price','Date':'date'}).to_csv(STOCK_FILE,index=False)

if not STOCK_FILE.exists():
    raise FileNotFoundError('Umieść prawdziwe dzienne notowania S&P 500 w stock_index_prices.csv '
                            '(kolumny Date/Close albo date/price), lub ustaw DOWNLOAD_MARKET=True.')
if not BTC_TRANSFER.exists():
    raise FileNotFoundError('Uruchom 14. Brakuje zamrożonego modelu transferowego BTC.')

stock=load_prices(STOCK_FILE)
stock=stock.loc[stock.date.between('2015-01-01','2024-12-31')].reset_index(drop=True)
print('Giełdowa próba:',len(stock),'sesji',stock.date.min(),stock.date.max())
if len(stock)<600:raise ValueError('Za mało sesji giełdowych do walidacji 2015–2024')

## 1. LPPLS dla indeksu akcji — bez współdzielenia cen BTC
Jeśli istnieją dopasowania stock_index_lppls_all_fits.csv, użyj ich.
Generator awaryjny wykorzystuje osobną kalibrację L-BFGS-B, więc NIE
jest identyczny z dawnym Nelder-Mead dla BTC — porównanie jest wtedy
analizą przenośności z dodatkowym ograniczeniem implementacyjnym.

In [ ]:
if not STOCK_FITS.exists():
    if not GENERATE_STOCK_LPPLS:
        raise FileNotFoundError('Brak stock_index_lppls_all_fits.csv. '
            'Przeprowadź estymację analogiczną do 00 na indeksie akcji, '
            'albo ustaw GENERATE_STOCK_LPPLS=True (generator w bubble_tools.py).')
    from bubble_tools import fit_stock_lppls
    stock_fits=fit_stock_lppls(stock,lookbacks=STOCK_FIT_LOOKBACKS,
        step=STOCK_FIT_STEP,n_starts=STOCK_FIT_RESTARTS,seed=SEED,
        max_windows=STOCK_FIT_MAX_WINDOWS)
    if STOCK_FIT_MAX_WINDOWS is not None:
        stock_fits.to_csv(OUT/'16_pilot_stock_fits_INCOMPLETE.csv',index=False)
        raise RuntimeError('Zapisano wyłącznie pilotaż. Usuń MAX_WINDOWS i przelicz pełne dopasowania.')
    stock_fits.to_csv(STOCK_FITS,index=False)
    print('Zapisano dopasowania LPPLS indeksu:',len(stock_fits))

features,feature_groups=build_daily_features(stock,STOCK_FITS)
features.to_csv(OUT/'16_stock_features.csv',index=False)
print('Dni z zaakceptowanym LPPLS:',int((features.lppls_n_accepted>0).sum()),
      'z',len(features))

## 2. Niezależne oznaczenia BSADF na indeksie (bez użycia modelu BTC)
Ta sama klasa pomocnicza metody; jej krytyczne wartości wynikają z
symulacji null, nie z ręcznego dostrajania do prognoz modelu.

In [ ]:
ref,ref_params=monte_carlo_bsadf_reference(stock,min_window=MIN_WINDOW,
    endpoint_step=END_STEP,start_step=START_STEP,reps=MC_REPS,
    quantile=.95,seed=SEED)
ref.to_csv(OUT/'16_stock_reference_bsadf.csv',index=False)
(OUT/'16_reference_settings.json').write_text(json.dumps(ref_params,indent=2),encoding='utf-8')

## 3. Przeniesienie ZAMROŻONEGO klasyfikatora BTC na indeks
Żaden parametr modelu, imputacji, skalowania ani próg nie jest uczony
na danych z indeksu. Indeks 2021–2024 jest nową domeną testową.

In [ ]:
bundle=joblib.load(BTC_TRANSFER)
assert bundle['feature_group']=='Price+LPPLS'
assert bundle['train_end']=='2019-12-31'
cols=bundle['feature_cols']
missing=set(cols)-set(features.columns)
if missing:raise ValueError('Brak cech w indeksie: '+str(missing))
data=features.merge(ref[['date','bubble_state_gsadf']],on='date',how='left',validate='one_to_one')
test=data.loc[data.date.dt.year.between(2021,2024)&data.bubble_state_gsadf.notna()].copy()
if test.empty:raise ValueError('Brak testu 2021–2024 na danych indeksu')
score=bundle['model'].predict_proba(test[cols])[:,1]
threshold=bundle['threshold']
yhat=(score>=threshold).astype(int)
test['score']=score;test['y_pred']=yhat
test['y']=test.bubble_state_gsadf.astype(int)
if test.bubble_state_gsadf.nunique()<2:
    raise RuntimeError('Cannot evaluate discrimination: the stock-index test contains only one reference class.')
res=classification_stats(test.y,yhat,score)
res.update({'ticker':TICKER,'train_domain':'BTC','test_domain':'stock index',
            'threshold':threshold,'threshold_status':bundle['threshold_status'],
            'prediction_start':str(test.date.min().date()),
            'prediction_end':str(test.date.max().date())})
results=pd.DataFrame([res])
results.to_csv(OUT/'16_crossmarket_transfer_metrics.csv',index=False)
test[['date','price','y','score','y_pred']].to_csv(OUT/'16_stock_transfer_predictions_OOS.csv',index=False)
display(results.round(3))

## 4. Wykres: rzeczywisty indeks, BSADF i alarmy modelu BTC

In [ ]:
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(15,7),sharex=True,
                           gridspec_kw={'height_ratios':[4,1]})
ax1.plot(stock.date,stock.price,color='black',lw=1,label=f'{TICKER} Close')
refpos=ref.loc[ref.bubble_state_gsadf.eq(1),'date']
for i,dt in enumerate(refpos):
    ax1.axvspan(dt,dt+pd.Timedelta(days=1),alpha=.22,color='steelblue',
                label='Stan BSADF' if i==0 else None)
for i,dt in enumerate(test.loc[test.y_pred.eq(1),'date']):
    ax1.axvspan(dt,dt+pd.Timedelta(days=1),alpha=.23,color='grey',
                label='Alarm transferowy BTC' if i==0 else None)
ax1.axvline(test.date.min(),color='navy',ls=':',label='Test OOS')
ax1.set(ylabel='Poziom indeksu',title='Walidacja międzyrynkowa: BTC XGBoost na indeksie akcji')
ax1.legend(loc='upper left');ax1.grid(alpha=.2)
ax2.step(test.date,test.y_pred,where='post',color='navy')
ax2.set(ylim=(-.1,1.1),yticks=[0,1],xlabel='Data',ylabel='Alarm')
ax2.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'16_01_stock_transfer_alarm_days.png',dpi=170);plt.show()

print('UWAGA: Jedna domena indeksowa nie dowodzi skuteczności na wszystkich stock markets.')
print('Zapisano:',OUT.resolve())